(r-comparisons-npiv)=

# Nonparametric IV in ModernDiD and R

This comparison checks whether {func}`~moderndid.npiv` and R's `npiv`
package return the same Engel curve for food. Racine and Christensen wrote
`npiv` to carry out the procedures of Chen, Christensen, and Kankanala
(2024), the paper that ModernDiD's estimator follows. We fit the curve for
the households of the {ref}`nonparametric IV example <example_npiv>` in both
languages. The page then compares the number of spline segments that the
data choose, the curve and its slope, and the uniform confidence bands
around them.

Every quantity that doesn't depend on bootstrap draws agrees to
floating-point precision, from the candidate dimensions to the curve, its
slope, and their standard errors. Both packages also choose the same number
of spline segments for the curve. The threshold of the selection test and
the critical values of the bands differ slightly, because each language
draws its own bootstrap multipliers. The same pattern holds when log total
spending is its own instrument, the regression case behind
{func}`~moderndid.cont_did` with `dose_est_method="cck"`.

The R package runs inside this Python session through `rpy2` and reads the
same data that ModernDiD fits. To rerun the comparison yourself, follow the
R setup in the section on {ref}`running the comparisons <r-comparisons-running>`.

:::{admonition} What this comparison checks
:class: note

The assertions require the two selections to match and hold the curves,
slopes, and standard errors to the common tolerance. The quantities that
rest on bootstrap draws appear side by side without an assertion.
:::

In [1]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

import moderndid as did

with ro.default_converter.context():
    r_npiv = importr("npiv")

pd.set_option("display.precision", 8)

Error importing in API mode: ImportError("dlopen(/Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): Library not loaded: /Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib\n  Referenced from: <05EE76C0-33BF-3D4C-BC56-84005241BC88> /Users/jordandeklerk/moderndid/.venv/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so\n  Reason: tried: '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file), '/Library/Frameworks/R.framework/Versions/4.5-arm64/Resources/lib/libRblas.dylib' (no such file)")
Trying to import in ABI mode.


## The same households for both fits

The 1,655 households that {func}`~moderndid.load_engel` returns come from
the 1995 British Family Expenditure Survey. The curve links `food`, the
share of spending that goes to food eaten at home, to `logexp`, the log of
total spending. The log earnings of the household head in `logwages` are the
instrument for total spending.

The `Engel95` data that come with R's `npiv` hold the same households.
Although the two copies list them in a different order, their values differ
by at most 5.3e-15. Passing ModernDiD's copy to R removes both differences
and gives the two fits identical inputs. Since few households sit at either
end of the spending range, we evaluate each package's curve on the
example's grid of 81 evenly spaced values of log spending from 4.5 to 6.5.

In [2]:
data = did.load_engel()
grid = np.linspace(4.5, 6.5, 81)

with (ro.default_converter + pandas2ri.converter).context():
    ro.globalenv["engel"] = ro.conversion.py2rpy(data.to_pandas())
with ro.default_converter.context():
    ro.globalenv["grid"] = ro.FloatVector(grid)

data.select("food", "logexp", "logwages").head()

food,logexp,logwages
f64,f64,f64
0.28026,3.609024,5.013565
0.379358,3.933002,2.71866
0.226277,4.064315,3.881564
0.167698,4.130275,4.900374
0.343115,4.259548,5.564099


## Fit the curve in each package

Both calls follow the specification that the nonparametric IV example sets
out. Each fit approximates the curve with a cubic spline in log spending and
leaves its number of segments to the data. Log earnings enter through
quartic splines with four times as many segments. The selection compares
its candidates at the same 81 grid points where the bands are drawn. The two
bands cover the curve and its slope at all of those points at once with 95
percent confidence. In each package, the selection and the bands reuse the
same 1,000 bootstrap draws from seed 7.

The dictionary `iv_spec` and the R function `fit_engel` hold everything but
the seed, since a later check refits both packages with other seeds. R's
summary prints the degree and the number of segments of each basis.

In [3]:
# The seed stays out of the specification because the seed check below varies it.
iv_spec = dict(
    yname="food",
    xname="logexp",
    wname="logwages",
    x_grid=grid,
    x_eval=grid,
    j_x_degree=3,
    k_w_degree=4,
    k_w_smooth=2,
    alpha=0.05,
    biters=1000,
)
py_fit = did.npiv(data, **iv_spec, seed=7)
print(f"segments: {py_fit.j_x_segments} for logexp, {py_fit.k_w_segments} for logwages")
print(f"critical values: {py_fit.cv:.4f} for the curve, {py_fit.cv_deriv:.4f} for the slope")

segments: 1 for logexp, 4 for logwages
critical values: 3.8678 for the curve, 3.8263 for the slope


In [4]:
with ro.default_converter.context():
    ro.r("""
        fit_engel <- function(seed) {
            set.seed(seed)
            npiv::npiv(
                Y = engel$food, X = engel$logexp, W = engel$logwages,
                X.grid = grid, X.eval = grid,
                J.x.degree = 3, K.w.degree = 4, K.w.smooth = 2,
                alpha = 0.05, boot.num = 1000, progress = FALSE
            )
        }
    """)
    r_fit = ro.r("fit_engel(7)")
    _ = ro.r["summary"](r_fit)

Call:
npiv.default(Y = engel$food, X = engel$logexp, W = engel$logwages, 
    X.eval = grid, X.grid = grid, alpha = 0.05, boot.num = 1000, 
    J.x.degree = 3, K.w.degree = 4, K.w.smooth = 2, progress = FALSE)

Nonparametric IV Model:

IV Regression Data: 1655 training points, and 81 evaluation points, in 1 endogenous variable(s)

B-spline degree for instruments:             4
B-spline segments for instruments:           4

B-spline degree for endogenous predictors:   3
B-spline segments for endogenous predictors: 1

Estimation time: 1.2 seconds



## Compare the dimension the data choose

Choosing the number of spline segments takes two steps in each package. The
first step uses no random draws and limits the candidates to those that the
sample size and the estimated strength of the instrument can support. The
second runs a bootstrap Lepski test and settles on the coarsest candidate
that no finer candidate rejects. A finer candidate rejects a coarser one
when the sup-t statistic between their curves exceeds 1.1 times
`theta_star`, a bootstrap quantile of the largest such statistic across all
pairs of candidates.

ModernDiD's result keeps the details of the selection in its `args` field.
Since R's `npiv` result keeps only the selected segments, the next cell
calls `npiv_choose_J`, the exported function that `npiv` runs for this step.
With seed 7, its draws repeat the ones inside the fit.

In [5]:
with ro.default_converter.context():
    ro.r("set.seed(7)")
    r_selection = ro.r("""
        npiv::npiv_choose_J(
            Y = engel$food, X = engel$logexp, W = engel$logwages, X.grid = grid,
            J.x.degree = 3, K.w.degree = 4, K.w.smooth = 2, boot.num = 1000, progress = FALSE
        )
    """)
    r_choice = {name: np.asarray(r_selection.rx2(name)) for name in r_selection.names}

# Each selection field of ModernDiD and the element of R's list that holds the same count.
fields = {
    "j_x_segments_set": "J.x.segments.set",
    "k_w_segments_set": "K.w.segments.set",
    "j_hat_max": "J.hat.max",
    "j_hat_n": "J.hat.n",
    "j_hat": "J.hat",
    "j_tilde": "J.tilde",
    "j_x_seg": "J.x.seg",
    "k_w_seg": "K.w.seg",
}


def selection_table(py_args, r_choice):
    """Set each selection result of ModernDiD beside the matching element of R's list."""
    table = pd.DataFrame(
        {
            "R element": list(fields.values()),
            "ModernDiD": [np.asarray(py_args[name]).tolist() for name in fields],
            # R stores these counts as doubles.
            "R": [r_choice[name].astype(int).squeeze().tolist() for name in fields.values()],
        },
        index=pd.Index(list(fields), name="ModernDiD field"),
    )
    table.loc["theta_star"] = [
        "theta.star",
        round(float(py_args["theta_star"]), 4),
        round(float(r_choice["theta.star"][0]), 4),
    ]
    return table


selection_table(py_fit.args, r_choice)

,R element,ModernDiD,R
ModernDiD field,,,
j_x_segments_set,J.x.segments.set,"[1, 2, 4, 8, 16]","[1, 2, 4, 8, 16]"
k_w_segments_set,K.w.segments.set,"[4, 8, 16, 32, 64]","[4, 8, 16, 32, 64]"
j_hat_max,J.hat.max,19,19
j_hat_n,J.hat.n,11,11
j_hat,J.hat,4,4
j_tilde,J.tilde,4,4
j_x_seg,J.x.seg,1,1
k_w_seg,K.w.seg,4,4
theta_star,theta.star,2.6046,2.5693


The two packages agree on every count that the table lists. Both compare
cubic splines with 1, 2, 4, 8, and 16 segments, the largest of them with 19
basis functions. Each would cut any choice above 8 segments, or 11 basis
functions, back to that point. The Lepski test settles on one segment in
each package, a single cubic polynomial with 4 basis functions. Since that
choice lies below the truncation point, it is the final selection.

The bootstrap draws enter the selection only through `theta_star`, 2.6046 in
ModernDiD and 2.5693 in R. That gap of 1.4 percent can't change the choice
here, because the largest sup-t statistic between one segment and a finer
candidate, 2.32 against 16 segments, stays well below either rejection
threshold, 2.87 in ModernDiD and 2.83 in R.

## Compare the curve and its slope

Once the dimension is fixed, the curve, its slope, and their pointwise
standard errors follow from the data without random draws. The table covers
them at the 81 grid points along with the four spline coefficients. For each
quantity it lists the largest absolute value in R beside the largest
absolute difference between the packages.

In [6]:
with ro.default_converter.context():
    r_values = {
        name: np.asarray(r_fit.rx2(name)).ravel()
        for name in ("h", "deriv", "asy.se", "deriv.asy.se", "beta")
    }
py_values = {
    "h": py_fit.h,
    "deriv": py_fit.deriv,
    "asy_se": py_fit.asy_se,
    "deriv_asy_se": py_fit.deriv_asy_se,
    "beta": np.ravel(py_fit.beta),
}
pd.DataFrame(
    {
        "R element": list(r_values),
        "largest absolute value in R": [np.max(np.abs(values)) for values in r_values.values()],
        "largest absolute difference": [
            np.max(np.abs(py - r))
            for py, r in zip(py_values.values(), r_values.values(), strict=True)
        ],
    },
    index=pd.Index(list(py_values), name="ModernDiD field"),
)

,R element,largest absolute value in R,largest absolute difference
ModernDiD field,,,
h,h,0.26139763,3.03368441e-14
deriv,deriv,0.09343341,8.31140712e-14
asy_se,asy.se,0.02042278,5.07580089e-15
deriv_asy_se,deriv.asy.se,0.06887576,3.74353326e-14
beta,beta,0.31794721,2.21267449e-13


No difference in the table reaches 3e-13, against values as large as 0.32.
Both packages solve the same two-stage least squares problem with the same
generalized inverse, a singular value decomposition that drops singular
values below the square root of machine precision times the largest. The
remaining gaps are rounding error, since each package evaluates the splines
with its own code and runs its matrix decompositions through its own linear
algebra library.

## Compare the bands

Each band is the estimate plus or minus a critical value times the pointwise
standard error. Since the estimates and standard errors agree, the bands of
the two packages can differ only through their critical values. The table
sets those critical values side by side and divides ModernDiD's half-width
by R's at each of the 81 grid points.

In [7]:
with ro.default_converter.context():
    r_cv = np.array([r_fit.rx2("cv")[0], r_fit.rx2("cv.deriv")[0]])
    r_upper = {name: np.asarray(r_fit.rx2(name)).ravel() for name in ("h.upper", "h.upper.deriv")}
py_cv = np.array([py_fit.cv, py_fit.cv_deriv])

# A ratio that stays the same at every point means the bands differ only by their critical values.
half_width_ratio = [
    (py_fit.h_upper - py_fit.h) / (r_upper["h.upper"] - r_values["h"]),
    (py_fit.h_upper_deriv - py_fit.deriv) / (r_upper["h.upper.deriv"] - r_values["deriv"]),
]
pd.DataFrame(
    {
        "ModernDiD critical value": py_cv,
        "R critical value": r_cv,
        "critical value ratio": py_cv / r_cv,
        "smallest half-width ratio": [ratio.min() for ratio in half_width_ratio],
        "largest half-width ratio": [ratio.max() for ratio in half_width_ratio],
    },
    index=pd.Index(["curve", "slope"], name="band"),
)

,ModernDiD critical value,R critical value,critical value ratio,smallest half-width ratio,largest half-width ratio
band,,,,,
curve,3.86778355,3.86432028,1.00089622,1.00089622,1.00089622
slope,3.82626627,3.85389643,0.99283059,0.99283059,0.99283059


ModernDiD's band for the curve is 0.09 percent wider than R's at every grid
point, since its critical value is 0.09 percent larger. Its band for the
slope is 0.72 percent narrower for the same reason. Both parts of each
critical value depend on the bootstrap draws. One part is a bootstrap
quantile of the band's largest sup-t statistic across the candidates with 1,
2, and 4 segments. The other is a margin for the data's choice of dimension,
`theta_star` times log log 4 for the 4 basis functions of the selected
curve.

:::{admonition} Seeds in each language
:class: note

Seed 7 reproduces each package's numbers from one run to the next. It can't
align the two packages, because NumPy and R turn a seed into normal draws
with different generators.
:::

When the multipliers come from R's draws instead, ModernDiD reproduces R's
`theta_star` and both critical values on this specification to within 3e-12
in relative terms. The reference tests in `tests/validation/test_r_npiv.py`
run the same check on other specifications and require the critical values
to agree to within 1e-8 to 1e-10.

(r-comparisons-npiv-seeds)=

### How far the critical values move between seeds

Whether a gap of 0.0035 in the curve's critical value is large depends on
how far that critical value moves between seeds within one package. The next
cell refits both packages with seeds 0 to 49 and summarizes the segments
they choose and the two critical values.

In [8]:
seeds = list(range(50))
py_runs = [did.npiv(data, **iv_spec, seed=seed) for seed in seeds]
py_seeds = pd.DataFrame(
    {
        "segments": [fit.j_x_segments for fit in py_runs],
        "curve critical value": [fit.cv for fit in py_runs],
        "slope critical value": [fit.cv_deriv for fit in py_runs],
    }
)
with ro.default_converter.context():
    ro.globalenv["seeds"] = ro.IntVector(seeds)
    r_runs = np.asarray(
        ro.r("""
            t(sapply(seeds, function(seed) {
                fit <- fit_engel(seed)
                c(fit$J.x.segments, fit$cv, fit$cv.deriv)
            }))
        """)
    )
r_seeds = pd.DataFrame(r_runs, columns=py_seeds.columns).astype({"segments": int})

seed_runs = pd.concat({"ModernDiD": py_seeds, "R": r_seeds}, names=["package", "seed"])
seed_runs.groupby("package").agg(
    {
        "segments": ["min", "max"],
        "curve critical value": ["mean", "std"],
        "slope critical value": ["mean", "std"],
    }
)

segments     curve critical value             slope critical value  \
               min max                 mean         std                 mean   
package                                                                        
ModernDiD        1   1           3.85081281  0.05472311           3.82384399   
R                1   1           3.85060873  0.04515042           3.82450425   

                       
                  std  
package                
ModernDiD  0.05326300  
R          0.04168607

In both packages, all 50 seeds lead the selection to a single segment. The
curve's critical value averages 3.8508 in ModernDiD and 3.8506 in R, against
standard deviations of 0.055 and 0.045 from seed to seed. Next to that
spread, the gap of 0.0035 at seed 7 is small. The slope's gap of 0.028 sits
within its own spread too, since its critical value averages 3.8238 and
3.8245 with standard deviations of 0.053 and 0.042. With averages this
close, the two bootstraps estimate the same critical values and differ only
in their draws.

## The regression special case

Passing log total spending as its own instrument turns both fits into least
squares regressions of the food share on cubic splines. In place of the
estimated strength of the instrument, the first selection step then uses a
factor that only the sample size sets, 1 for these 1,655 households. Without
an instrument to limit them, the candidates reach finer splines than in the
IV fit. We leave out `x_grid` here, as the example does when it checks the
full spending range. Each selection then compares its candidates at 50
points from the smallest to the largest log spending, 3.61 to 7.43.

In [9]:
py_regression = did.npiv(data, **(iv_spec | {"wname": "logexp", "x_grid": None}), seed=7)
print(f"segments: {py_regression.j_x_segments} for logexp in ModernDiD")

with ro.default_converter.context():
    ro.r("set.seed(7)")
    r_regression = ro.r("""
        npiv::npiv(
            Y = engel$food, X = engel$logexp, W = engel$logexp, X.eval = grid,
            J.x.degree = 3, alpha = 0.05, boot.num = 1000, progress = FALSE
        )
    """)
    _ = ro.r["summary"](r_regression)

    # npiv gives the instrument basis the regressor's degree and segments before it selects.
    ro.r("set.seed(7)")
    r_regression_selection = ro.r("""
        npiv::npiv_choose_J(
            Y = engel$food, X = engel$logexp, W = engel$logexp,
            J.x.degree = 3, K.w.degree = 3, K.w.smooth = 0, boot.num = 1000, progress = FALSE
        )
    """)
    r_regression_choice = {
        name: np.asarray(r_regression_selection.rx2(name)) for name in r_regression_selection.names
    }

segments: 64 for logexp in ModernDiD
Call:
npiv.default(Y = engel$food, X = engel$logexp, W = engel$logexp, 
    X.eval = grid, alpha = 0.05, boot.num = 1000, J.x.degree = 3, 
    progress = FALSE)

Nonparametric IV Model:

IV Regression Data: 1655 training points, and 81 evaluation points, in 1 endogenous variable(s)

B-spline degree for instruments:             3
B-spline segments for instruments:           64

B-spline degree for endogenous predictors:   3
B-spline segments for endogenous predictors: 64

Estimation time: 4.2 seconds



In [10]:
with ro.default_converter.context():
    r_regression_cv = [float(r_regression.rx2("cv")[0]), float(r_regression.rx2("cv.deriv")[0])]
regression = selection_table(py_regression.args, r_regression_choice)
regression.loc["cv"] = ["cv", round(py_regression.cv, 4), round(r_regression_cv[0], 4)]
regression.loc["cv_deriv"] = [
    "cv.deriv",
    round(py_regression.cv_deriv, 4),
    round(r_regression_cv[1], 4),
]
regression

,R element,ModernDiD,R
ModernDiD field,,,
j_x_segments_set,J.x.segments.set,"[1, 2, 4, 8, 16, 32, 64, 128]","[1, 2, 4, 8, 16, 32, 64, 128]"
k_w_segments_set,K.w.segments.set,"[1, 2, 4, 8, 16, 32, 64, 128]","[1, 2, 4, 8, 16, 32, 64, 128]"
j_hat_max,J.hat.max,131,131
j_hat_n,J.hat.n,67,67
j_hat,J.hat,131,131
j_tilde,J.tilde,67,67
j_x_seg,J.x.seg,64,64
k_w_seg,K.w.seg,64,64
theta_star,theta.star,3.3763,3.4129


Both regressions compare eight candidates over the full range, from 1 to 128
segments. Since each coarser candidate's sup-t statistic against some finer
one exceeds 1.1 times `theta_star`, the Lepski test in both packages picks
the largest, 128 segments with 131 basis functions. Even for 32 segments,
the closest call, the largest statistic of 4.51 exceeds both rejection
thresholds, 3.71 in ModernDiD and 3.75 in R. Both packages then cut the
choice back to the second-largest candidate, 64 segments or 67 basis
functions. As in the IV fit, `theta_star` and the two critical values differ
only through the draws, here by 0.9 to 1.4 percent. With R's draws,
ModernDiD reproduces all three to within 1e-13 in relative terms.

Chen, Christensen, and Kankanala (2024) keep the Lepski choice in their
regression procedure (Appendix A, step 3). Both implementations instead
truncate regressions the way the IV procedure truncates IV fits. In `npiv`
0.1.3, lines 961 and 962 of
[`R/npiv.R`](https://github.com/cran/npiv/blob/0.1.3/R/npiv.R#L961-L962)
take the smaller of the Lepski choice and the second-largest candidate.
{func}`~moderndid.npiv_j` takes the same minimum for every fit, regressions
included. Under the paper's rule, both regressions would keep the Lepski
choice of 128 segments.

:::{admonition} What agreement with npiv covers
:class: note

Matching `npiv` confirms the procedure as its authors implemented it rather
than every step of the paper. Beyond the regression cut above, the two
packages also share a band without the bias envelope that the paper adds
when truncation sets the dimension. The
{ref}`background page <background-npiv>` explains where the bands depart
from the paper and what that means for coverage.
:::

## Assert the agreement

The last cell requires the two selections to match field by field in each
fit. It holds every curve and slope estimate within
`1e-6 + 1e-5 * abs(R estimate)` and every standard error within
`1e-6 + 1e-3 * abs(R SE)`, the bounds that the
{ref}`staggered DiD comparison <r-comparisons-staggered>` also uses. Its
table then lists the largest gap for the IV fit and for the regression.

In [11]:
with ro.default_converter.context():
    r_regression_values = {
        name: np.asarray(r_regression.rx2(name)).ravel()
        for name in ("h", "deriv", "asy.se", "deriv.asy.se")
    }

rows = []
for label, py_result, r_result, r_selected in (
    ("IV fit", py_fit, r_values, r_choice),
    ("regression", py_regression, r_regression_values, r_regression_choice),
):
    for name, r_name in fields.items():
        np.testing.assert_array_equal(
            np.asarray(py_result.args[name]), r_selected[r_name].squeeze()
        )
    np.testing.assert_allclose(py_result.h, r_result["h"], rtol=1e-5, atol=1e-6)
    np.testing.assert_allclose(py_result.deriv, r_result["deriv"], rtol=1e-5, atol=1e-6)
    np.testing.assert_allclose(py_result.asy_se, r_result["asy.se"], rtol=1e-3, atol=1e-6)
    np.testing.assert_allclose(
        py_result.deriv_asy_se, r_result["deriv.asy.se"], rtol=1e-3, atol=1e-6
    )
    rows.append(
        {
            "comparison": label,
            "largest curve difference": np.max(np.abs(py_result.h - r_result["h"])),
            "largest slope difference": np.max(np.abs(py_result.deriv - r_result["deriv"])),
            "largest standard error difference": max(
                np.max(np.abs(py_result.asy_se - r_result["asy.se"])),
                np.max(np.abs(py_result.deriv_asy_se - r_result["deriv.asy.se"])),
            ),
        }
    )
pd.DataFrame(rows)

,comparison,largest curve difference,largest slope difference,largest standard error difference
0,IV fit,3.03368441e-14,8.31140712e-14,3.74353326e-14
1,regression,1.79856130e-14,1.07092113e-12,5.19584376e-14


## Versions used

The table records the versions of the software that produced the saved
output above.

In [12]:
from importlib.metadata import version
from platform import python_version

with ro.default_converter.context():
    r_version = str(ro.r("R.version.string")[0])
    npiv_version = str(ro.r('as.character(packageVersion("npiv"))')[0])
pd.DataFrame(
    {
        "component": ["Python", "ModernDiD", "rpy2", "R", "npiv"],
        "version": [
            python_version(), version("moderndid"), version("rpy2"), r_version, npiv_version
        ],
    }
)

,component,version
0,Python,3.12.10
1,ModernDiD,0.2.0
2,rpy2,3.6.4
3,R,R version 4.3.3 (2024-02-29)
4,npiv,0.1.3
